In [ ]:
using WAVI
using MAT, FileIO  # For loading .mat files
using Printf, Plots

In [ ]:
# Grid
nx = 106#162 # use the clipped grid
ny = 123#177 # use the clipped grid
nσ = 26 # tried with less than 26, but could not reach sigma=1.0 and got error. seems to be given by contructor but cannot identify where exactly
x0 = -1.812e6#-2260000.0
y0 = -860000.0#-1292000.0
dx = 8000.0
dy = 8000.0

# Paths: static fields from original setup, evolved fields from inversion output
static_basepath = "/media/thanasis/DATA_BACKUP/BAS_laptop/repos/WAVIConstructor.jl/examples/outputs/amundsen_verification_v2/"
inversion_output_path = "/media/thanasis/DATA_BACKUP/BAS_laptop/repos/WAVI.jl/example_drivers/Rosie_examples/test_amundsen_verification_8km_v2/"  # UPDATE THIS: path to where inversion .mat files are saved
inversion_iteration = 25  # UPDATE THIS: which iteration to load (last iteration = 25 for your setup)

In [ ]:
# ============================================================================
# Load static fields (these don't change during inversion)
# ============================================================================
hm=Array{Float64}(undef,nx,ny);
read!(joinpath(static_basepath, "h_mask.bin"),hm);
# hm = ntoh.(h_mask);
hm = map.(Bool, round.(Int, hm));

u_iszero=Array{Float64}(undef,nx+1,ny);
read!(joinpath(static_basepath, "u_iszero.bin"),u_iszero);
# u_iszero.=ntoh.(u_iszero);
u_iszero = map.(Bool, round.(Int, u_iszero));

v_iszero=Array{Float64}(undef,nx,ny+1);
read!(joinpath(static_basepath,"v_iszero.bin"),v_iszero);
# v_iszero.=ntoh.(v_iszero);
v_iszero = map.(Bool, round.(Int, v_iszero));

sigma_grid=Array{Float64}(undef,nσ);
read!(joinpath(static_basepath,"sigma_grid.bin"),sigma_grid);
# sigma_grid.=ntoh.(sigma_grid);
println(sigma_grid)

bed=Array{Float64}(undef,nx,ny);
read!(joinpath(static_basepath, "bed.bin"),bed);
# bed.=ntoh.(bed);

temp=Array{Float64}(undef,nx,ny,nσ);
read!(joinpath(static_basepath, "temps.bin"),temp);
# temp.=ntoh.(temp);
;

In [ ]:
heatmap(temp[:,:,12]')

In [ ]:
# ============================================================================
# Create grid (after loading masks and sigma_grid)
# ============================================================================
grid = Grid(nx = nx,
            ny = ny,
            nσ = nσ,
            x0 = x0,
            y0 = y0,
            dx = dx,
            dy = dy,
            h_mask = hm,
            u_iszero = u_iszero,
            v_iszero = v_iszero,
            σ = sigma_grid)

# ============================================================================
# Load evolved fields from inversion .mat output
# ============================================================================
# Construct filename for the inversion iteration (format: outfile0000000025.mat)
mat_filename = @sprintf("outfile%010d.mat", inversion_iteration)
mat_filepath = joinpath(inversion_output_path, mat_filename)

if !isfile(mat_filepath)
    error("Inversion output file not found: $mat_filepath")
end

println("Loading inversion output from: $mat_filepath")
mat_data = matread(mat_filepath) #load(mat_filepath)  # Loads a dictionary of arrays

# Extract evolved fields from .mat file
# Note: Field names in .mat file match the output names from your inversion driver
h = mat_data["h"]  # Thickness (nx, ny)
u = mat_data["u"]  # U velocity (nx+1, ny) 
v = mat_data["v"]  # V velocity (nx, ny+1)
beta = mat_data["beta"]  # Beta field (nx, ny)
grounded_fraction = mat_data["grounded_fraction"];  # Grounded fraction (nx, ny)

In [ ]:
heatmap(grounded_fraction')

In [ ]:

# damage = mat_data["damage"]  # Damage field (nσ, ny, nx) - check orientation!
# viscosity = mat_data["eta"]  # 3D viscosity (nσ, ny, nx) - check orientation! (eta in .mat, but called viscosity here to match original)

# # Check and fix orientation if needed
# # WAVI expects initial_damage and initial_viscosity to be (nx, ny, nσ)
# # But .mat files may store them in different orientations due to MATLAB/HDF5 transposition
# if size(damage) == (nx, ny, nσ)
#     # Already correct orientation: (nx, ny, nσ) - WAVI's expected format
#     damage = damage
#     viscosity = viscosity
# elseif size(damage) == (nσ, ny, nx)
#     # Permute from (nσ, ny, nx) -> (nx, ny, nσ)
#     damage = permutedims(damage, (3, 2, 1))
#     viscosity = permutedims(viscosity, (3, 2, 1))
# elseif size(damage) == (ny, nx, nσ)
#     # Permute from (ny, nx, nσ) -> (nx, ny, nσ)
#     damage = permutedims(damage, (2, 1, 3))
#     viscosity = permutedims(viscosity, (2, 1, 3))
# else
#     error("Unexpected damage shape: $(size(damage)). Expected (nx, ny, nσ), (nσ, ny, nx), or (ny, nx, nσ)")
# end

# load damage and viscocity from checkpoint .jld2
checkpoint_file = load(inversion_output_path*"PChkpt_0000000005.jld2")
fields_3d = checkpoint_file["inversion_simulation"].model.fields.g3d

damage = fields_3d.Φ
viscosity = fields_3d.η

In [ ]:
heatmap(damage[:,:,1]')

In [ ]:
heatmap(viscosity[:,:,10]')

In [ ]:

# ============================================================================
# Calculate weertman_c from beta (full calculation)
# ============================================================================
# Default WAVI parameter values (if not specified in your inversion)
weertman_reg_speed = 1.0e-5  # Default from WAVI
weertman_m = 3.0  # Default from WAVI
density_ice = 918.0  # Default from WAVI
density_ocean = 1028.0  # Default from WAVI
sea_level_wrt_geoid = 0.0  # Default from WAVI

# # Load bed_speed and haf from .mat file (required for weertman_c calculation)
# if !haskey(mat_data, "bed_speed")
#     error("bed_speed not found in .mat file. Add 'bed_speed = model.fields.gh.bed_speed' to your inversion outputs.")
# end
# if !haskey(mat_data, "haf")
#     error("haf not found in .mat file. Add 'haf = model.fields.gh.haf' to your inversion outputs.")
# end

bed_speed = checkpoint_file["inversion_simulation"].model.fields.gh.bed_speed#mat_data["bed_speed"]
haf = checkpoint_file["inversion_simulation"].model.fields.gh.haf #mat_data["haf"]
println("Loaded bed_speed and haf from .jld2 file")
#println("Loaded bed_speed and haf from .mat file")

# Create mask (same as h_mask)
# mask is a boolean array (nx, ny) where true = ice exists, false = no ice
mask = hm

# Calculate weertman_c from beta (full calculation)
# This inverts the Weertman sliding law: β = weertman_c * (sqrt(bed_speed² + reg_speed²))^(1/m - 1)
# To get: weertman_c = β / (sqrt(bed_speed² + reg_speed²))^(1/m - 1)
weertman_c_newly_grounded = 1.0e4
weertman_c = zeros(size(beta))  # Initialize array for all grid cells (nx, ny)

# Calculate weertman_c for ice-covered cells only
# mask is boolean: true = ice, false = no ice
# weertman_c[mask] uses boolean indexing to select only ice-covered cells (returns 1D array)
# beta[mask] selects beta values at ice-covered cells only
# bed_speed[mask] selects bed_speed values at ice-covered cells only
# The calculation: weertman_c = β / (sqrt(bed_speed² + reg_speed²))^(1/m - 1)
weertman_c[mask] = beta[mask] ./ 
    ((sqrt.(bed_speed[mask].^2 .+ weertman_reg_speed^2)).^(1.0/weertman_m .- 1.0))
# This assigns weertman c values only to ice-covered cells, leaving non-ice cells as zero

# Adjust for partially grounded cells
weertman_c_adjusted = zeros(size(beta))
aground = (haf .>= 0)
f1 = findall(aground .== 1)
f2 = findall(aground .== 0)

weertman_c_adjusted[f1] .= weertman_c[f1] ./ grounded_fraction[f1]
weertman_c_adjusted[f2] .= weertman_c_newly_grounded

# Remove any NaNs
weertman_c_adjusted[isnan.(weertman_c_adjusted) .& .!(weertman_c_adjusted .∈ Ref(mask))] .= -9999.0

weertman_c = weertman_c_adjusted
println("Calculated weertman_c from beta")

In [ ]:
# ============================================================================
# Build model
# ============================================================================

#Initial conditions from inversion output

initial_conditions = InitialConditions(
    initial_thickness = h,
    initial_viscosity = viscosity,
    initial_temperature = temp,
    initial_damage = damage,
    initial_u_veloc = u,
    initial_v_veloc = v
)


In [ ]:

# Accumulation rate (static, from original setup)
accumulation_rate=Array{Float64}(undef,nx,ny);
read!(joinpath(static_basepath, "accumulation_data.bin"),accumulation_rate);
#accumulation_rate.=ntoh.(accumulation_rate);

# For relaxation runs, we do not evolve shelves
evolveShelves = false

params = Params(accumulation_rate = accumulation_rate,
                weertman_c = weertman_c,
                evolveShelves = evolveShelves)

# Use QuadraticMeltRate (update γT value as needed)
#melt_rate = QuadraticMeltRate(γT = 5.3e-5)  # UPDATE THIS: set your melt rate parameter

#solver parameters
solver_params = SolverParams(maxiter_picard = 2)

#parallel_spec = SharedMemorySpec(ngridsx = 2, ngridsy=4, overlap=1, niterations=1)

model = Model(grid = grid, 
              bed_elevation = bed, 
              initial_conditions = initial_conditions, 
              params=params, 
             #parallel_spec = parallel_spec, 
              solver_params = solver_params)
              #melt_rate = melt_rate  # Commented out - no melt for relaxation

#
#timestepping parameters
niter0 = 0
step_thickness = true
dt = 0.1
end_time = 20.
chkpt_freq = 1.
pchkpt_freq = 10.
timestepping_params = TimesteppingParams(
    niter0 = niter0,
    dt = dt,
    end_time = end_time,
    chkpt_freq = chkpt_freq,
    pchkpt_freq = pchkpt_freq,
    step_thickness = step_thickness)

#output parameters
# Include all fields needed for forward runs, matching inversion outputs
outputs = (h = model.fields.gh.h,
           u = model.fields.gu.u,                    # U-grid velocity (for initial_u_veloc)
           uh = model.fields.gh.u,                   # H-grid u velocity (depth-averaged)
           v = model.fields.gv.v,                    # V-grid velocity (for initial_v_veloc)
           vh = model.fields.gh.v,                   # H-grid v velocity (depth-averaged)
           grounded_fraction = model.fields.gh.grounded_fraction,
           beta = model.fields.gh.β,                 # For calculating weertman_c
           preBfactor = model.fields.gh.preBfactor,
           damage = model.fields.g3d.Φ,              # 3D damage field
           eta_av = model.fields.gh.ηav,             # Depth-averaged viscosity
           eta = model.fields.g3d.η,                 # Full 3D viscosity (for initial_viscosity)
           bed_speed = model.fields.gh.bed_speed,    # Needed for weertman_c calculation
           haf = model.fields.gh.haf,                # Height above flotation, needed for weertman_c calculation
           temperature = model.fields.g3d.θ,         # 3D temperature field (for initial_temperature)
           b = model.fields.gh.b)                    # Bed elevation (static, but useful for reference)
output_freq = 2.
output_params = OutputParams(outputs = outputs,
                output_freq = output_freq,
                output_format = "mat",
                dump_vel = true,
                zip_format = "nc",
                output_path = "./amundsen_relaxation_out/")

# assemble the simulation
simulation = Simulation(model = model, 
                       timestepping_params = timestepping_params, 
                       output_params = output_params)


In [ ]:
#perform the simulation
run_simulation!(simulation)

In [ ]:
f_f = simulation.model.fields.g3d.η;
f_i = checkpoint_file["inversion_simulation"].model.fields.g3d.η;
f_diff = f_f .- f_i;


In [ ]:
l = 1

p1 = heatmap(f_i[:,:,l]')
p2 = heatmap(f_f[:,:,l]')
p3 = heatmap(f_diff[:,:,l]')

plot(p1, p2, p3, layout = (1, 3), size = (1300, 400))